# Smart-Demand: Sales Volume Prediction System

**Machine Learning | Final Project**

**Team:**
- Christopher Setyawan (2802459670)
- Klaus Anson (2802459361)
- Owen Aldrich Setiawan (2802463314)
- Lawrance Velasques (2802477080)
- Gracias Kumara Winata (2802459683)

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Task:** Regression: predict quantity sold per product per month  
**Main Model:** Random Forest Regressor  
**Baseline:** Linear Regression

**Notebook Structure:**
1. Setup & Library Import
2. Load Dataset
3. Data Integration
4. Data Cleaning
5. Feature Engineering
6. Exploratory Data Analysis
7. Train-Test Split
8. Model Training
9. Evaluation & Comparison
10. Feature Importance
11. Save Model

## 1. Setup & Library Import

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

import sklearn
print(f'pandas       : {pd.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print('Libraries loaded successfully.')

## 2. Load Dataset

We use 6 out of 9 available CSV files from the Olist dataset. The geolocation, customers, and sellers tables are excluded since the model focuses on product-level prediction, not geographic analysis.

If accessing the dataset from a shared Drive folder, right-click the folder and choose Add shortcut to My Drive so everyone on the team can use the same path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to match your actual folder path in Google Drive
DATA_PATH = '/content/drive/MyDrive/dataset/'

print(f'Data path: {DATA_PATH}')
print('CSV files found in folder:')
for f in sorted(os.listdir(DATA_PATH)):
    if f.endswith('.csv'):
        print(f'  {f}')

In [ ]:
df_orders     = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
df_items      = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
df_products   = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
df_reviews    = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
df_payments   = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
df_cat_transl = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('Dataset shapes loaded:')
print(f'  orders      : {df_orders.shape}')
print(f'  items       : {df_items.shape}')
print(f'  products    : {df_products.shape}')
print(f'  reviews     : {df_reviews.shape}')
print(f'  payments    : {df_payments.shape}')
print(f'  cat_transl  : {df_cat_transl.shape}')

## 3. Data Integration

All 6 tables are joined into a single master dataset, then aggregated at the product-month level. Each row represents one product in one specific month.

The join follows this order: items + orders (purchase date) + reviews (rating) + payments (actual payment) + products (category) + category translation. After joining, only delivered orders are kept, then data is aggregated by product and month.

In [ ]:
# Average review score per order
review_avg = (
    df_reviews
    .groupby('order_id')['review_score']
    .mean()
    .reset_index()
    .rename(columns={'review_score': 'product_rating'})
)

# Total payment value per order
payment_per_order = (
    df_payments
    .groupby('order_id')['payment_value']
    .sum()
    .reset_index()
    .rename(columns={'payment_value': 'total_payment'})
)

# Build master from items
master = df_items.copy()

master = master.merge(
    df_orders[['order_id', 'order_purchase_timestamp', 'order_status']],
    on='order_id', how='left'
)
master = master.merge(review_avg, on='order_id', how='left')
master = master.merge(payment_per_order, on='order_id', how='left')
master = master.merge(
    df_products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)
master = master.merge(df_cat_transl, on='product_category_name', how='left')

# Use English category name, fall back to Portuguese if translation not available
master['category'] = (
    master['product_category_name_english']
    .fillna(master['product_category_name'])
    .fillna('unknown')
)

# Keep only delivered orders
master = master[master['order_status'] == 'delivered'].copy()

print(f'Master dataset shape after join: {master.shape}')

In [ ]:
# Parse timestamps and extract year/month
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
master['year']  = master['order_purchase_timestamp'].dt.year
master['month'] = master['order_purchase_timestamp'].dt.month

# Total item price per order (needed later for discount_rate calculation)
order_item_total = (
    master.groupby('order_id')['price']
    .sum()
    .reset_index()
    .rename(columns={'price': 'item_total'})
)
master = master.merge(order_item_total, on='order_id', how='left')

# Aggregate to product-month level
df_agg = (
    master
    .groupby(['product_id', 'category', 'year', 'month'])
    .agg(
        quantity_sold  = ('order_item_id', 'count'),
        price_final    = ('price', 'mean'),
        freight_value  = ('freight_value', 'mean'),
        product_rating = ('product_rating', 'mean'),
        avg_item_total = ('item_total', 'mean'),
        avg_payment    = ('total_payment', 'mean')
    )
    .reset_index()
)

print(f'Aggregated dataset shape: {df_agg.shape}')
display(df_agg.head(5))

## 4. Data Cleaning

In [ ]:
print('Missing values before cleaning:')
print(df_agg.isnull().sum())
print(f'\nTotal rows: {len(df_agg):,}')

In [ ]:
df_clean = df_agg.copy()

# Fill missing rating with median per category, then overall median as fallback
median_by_cat = df_clean.groupby('category')['product_rating'].transform('median')
df_clean['product_rating'] = df_clean['product_rating'].fillna(median_by_cat)
df_clean['product_rating'] = df_clean['product_rating'].fillna(df_clean['product_rating'].median())

# Drop rows where price or freight is missing
df_clean = df_clean.dropna(subset=['price_final', 'freight_value'])

# Fill missing payment with item_total (assume no discount if payment data not available)
df_clean['avg_payment'] = df_clean['avg_payment'].fillna(df_clean['avg_item_total'])

# Remove extreme outliers in target using 3x IQR (conservative cutoff)
q1  = df_clean['quantity_sold'].quantile(0.25)
q3  = df_clean['quantity_sold'].quantile(0.75)
iqr = q3 - q1
df_clean = df_clean[df_clean['quantity_sold'] <= q3 + 3 * iqr]

print(f'Rows after cleaning: {len(df_clean):,}')
print('Missing values remaining:')
print(df_clean.isnull().sum())

## 5. Feature Engineering

Features constructed for this model:

- price_final - average product price per month
- discount_rate - discount ratio derived from payment vs item total
- freight_value - average shipping cost per month
- product_rating - average customer review score
- category_encoded - label-encoded product category
- seasonality_index - normalized monthly demand index (0 to 1)
- last_month_sales - units sold the previous month (lag-1)

In [ ]:
df_feat = df_clean.copy()

# Feature 1: Discount Rate
# How much the actual payment is below the listed item total (clipped at 0)
# If payment exceeds item total (e.g. installment fees), discount is treated as 0
df_feat['discount_rate'] = (
    (df_feat['avg_item_total'] - df_feat['avg_payment'])
    / df_feat['avg_item_total'].replace(0, np.nan)
).clip(lower=0).fillna(0)

print('Discount rate summary:')
print(df_feat['discount_rate'].describe().round(4))
print(f'\nRows with actual discount (> 0): {(df_feat["discount_rate"] > 0).sum():,}')

In [ ]:
# Feature 2: Seasonality Index
# Average monthly sales normalized to range [0, 1]
monthly_avg  = df_feat.groupby('month')['quantity_sold'].mean()
monthly_norm = (monthly_avg - monthly_avg.min()) / (monthly_avg.max() - monthly_avg.min())
df_feat['seasonality_index'] = df_feat['month'].map(monthly_norm)

print('Seasonality index per month (0 = slowest, 1 = busiest):')
print(monthly_norm.round(3).to_string())

In [ ]:
# Feature 3: Last Month Sales (Lag-1)
df_feat = df_feat.sort_values(['product_id', 'year', 'month'])
df_feat['last_month_sales'] = (
    df_feat.groupby('product_id')['quantity_sold']
    .shift(1)
    .fillna(0)
)

# Feature 4: Category Encoded
label_enc = LabelEncoder()
df_feat['category_encoded'] = label_enc.fit_transform(df_feat['category'])

print(f'Total categories encoded: {len(label_enc.classes_)}')
print('Feature engineering complete.')

In [ ]:
FEATURES = [
    'price_final',
    'discount_rate',
    'freight_value',
    'product_rating',
    'category_encoded',
    'seasonality_index',
    'last_month_sales'
]
TARGET = 'quantity_sold'

print(f'Input features  (X) : {FEATURES}')
print(f'Target variable (Y) : {TARGET}')
print(f'\nFinal dataset shape : {df_feat[FEATURES + [TARGET]].shape}')
display(df_feat[FEATURES + [TARGET]].describe().round(3))

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# 6.1 Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_feat[TARGET], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Quantity Sold', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity Sold')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df_feat[TARGET]), bins=50, color='cornflowerblue', edgecolor='white')
axes[1].set_title('Distribution of log(Quantity Sold + 1)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Quantity Sold + 1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_01_target_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Skewness : {df_feat[TARGET].skew():.3f}')
print(f'Mean     : {df_feat[TARGET].mean():.2f}')
print(f'Median   : {df_feat[TARGET].median():.2f}')

In [ ]:
# 6.2 Correlation heatmap
corr_matrix = df_feat[FEATURES + [TARGET]].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 10}
)
plt.title('Correlation Heatmap — Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_02_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.3 Top 15 categories by total sales
top_cats = (
    df_feat.groupby('category')['quantity_sold']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
colors = plt.cm.Blues(np.linspace(0.35, 0.85, 15))
plt.barh(top_cats.index[::-1], top_cats.values[::-1], color=colors[::-1])
plt.title('Top 15 Product Categories by Total Sales', fontsize=13, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.tight_layout()
plt.savefig('eda_03_top_categories.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.4 Monthly sales trend
monthly_sales = (
    df_feat.groupby(['year', 'month'])['quantity_sold']
    .sum()
    .reset_index()
)
monthly_sales['period'] = pd.to_datetime(
    monthly_sales.assign(day=1)[['year', 'month', 'day']]
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['period'], monthly_sales['quantity_sold'],
         marker='o', color='steelblue', linewidth=2, markersize=5)
plt.fill_between(monthly_sales['period'], monthly_sales['quantity_sold'],
                 alpha=0.15, color='steelblue')
plt.title('Monthly Sales Trend', fontsize=13, fontweight='bold')
plt.xlabel('Period')
plt.ylabel('Total Quantity Sold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_04_monthly_trend.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.5 Discount rate analysis
discounted = df_feat[df_feat['discount_rate'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(discounted['discount_rate'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Discount Rate Distribution\n(Orders with discount only)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('Frequency')

axes[1].scatter(df_feat['discount_rate'], df_feat['quantity_sold'],
                alpha=0.25, color='cornflowerblue', s=10)
axes[1].set_title('Discount Rate vs Quantity Sold', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Quantity Sold')

plt.suptitle('Discount Rate Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_05_discount_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Rows with discount > 0  : {len(discounted):,} ({len(discounted)/len(df_feat)*100:.1f}%)')
print(f'Average discount rate   : {discounted["discount_rate"].mean():.4f}')

In [ ]:
# 6.6 Price and rating vs target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_feat['price_final'], df_feat['quantity_sold'],
                alpha=0.25, color='steelblue', s=10)
axes[0].set_title('Price vs Quantity Sold', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Price Final')
axes[0].set_ylabel('Quantity Sold')

axes[1].scatter(df_feat['product_rating'], df_feat['quantity_sold'],
                alpha=0.25, color='cornflowerblue', s=10)
axes[1].set_title('Product Rating vs Quantity Sold', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Product Rating')
axes[1].set_ylabel('Quantity Sold')

plt.suptitle('Feature Relationships with Target', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_06_feature_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.7 EDA summary stats
print('EDA Summary')
print('-' * 45)
print(f'Total records         : {len(df_feat):,}')
print(f'Unique products       : {df_feat["product_id"].nunique():,}')
print(f'Unique categories     : {df_feat["category"].nunique():,}')
print(f'Date range            : {df_feat["year"].min()} - {df_feat["year"].max()}')
print(f'Mean quantity sold    : {df_feat[TARGET].mean():.2f}')
print(f'Median quantity sold  : {df_feat[TARGET].median():.2f}')
print(f'Max quantity sold     : {df_feat[TARGET].max():.0f}')
print(f'Mean price            : {df_feat["price_final"].mean():.2f}')
print(f'Mean rating           : {df_feat["product_rating"].mean():.2f} / 5.0')
print(f'Mean discount rate    : {df_feat["discount_rate"].mean():.4f}')

## 7. Train-Test Split

In [ ]:
X = df_feat[FEATURES].values
y = df_feat[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Train : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)')

## 8. Model Training

In [ ]:
# Baseline: Linear Regression
print('Training Linear Regression (baseline)...')
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print('Done.')

In [ ]:
# Main model: Random Forest Regressor
print('Training Random Forest Regressor...')
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('Done.')

## 9. Evaluation & Comparison

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)
    results = {
        'Model'      : model_name,
        'Train MAE'  : mean_absolute_error(y_train, train_pred),
        'Test MAE'   : mean_absolute_error(y_test, test_pred),
        'Train RMSE' : np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE'  : np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train R2'   : r2_score(y_train, train_pred),
        'Test R2'    : r2_score(y_test, test_pred),
    }
    return results, test_pred

lr_results, lr_pred = evaluate_model(lr_model, X_train, X_test, y_train, y_test, 'Linear Regression')
rf_results, rf_pred = evaluate_model(rf_model, X_train, X_test, y_train, y_test, 'Random Forest')

results_df = pd.DataFrame([lr_results, rf_results]).set_index('Model')
print('Model Comparison:')
display(results_df.round(4))

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred, name, color in zip(
    axes,
    [lr_pred, rf_pred],
    ['Linear Regression (Baseline)', 'Random Forest'],
    ['tomato', 'steelblue']
):
    ax.scatter(y_test, pred, alpha=0.25, color=color, s=10)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f'{name}\nR2 = {r2_score(y_test, pred):.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Quantity Sold')
    ax.set_ylabel('Predicted Quantity Sold')
    ax.legend()

plt.suptitle('Actual vs Predicted — Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_01_actual_vs_predicted.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Residual analysis for Random Forest
residuals = y_test - rf_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(rf_pred, residuals, alpha=0.25, color='steelblue', s=10)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Plot — Random Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual (Actual - Predicted)')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Residual Distribution — Random Forest', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('eval_02_residuals.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature'    : FEATURES,
    'importance' : rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
bar_colors = plt.cm.Blues(np.linspace(0.3, 0.85, len(FEATURES)))
plt.barh(importance_df['feature'], importance_df['importance'], color=bar_colors)
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print('Feature importance (highest to lowest):')
display(importance_df.sort_values('importance', ascending=False).round(4))

## 11. Save Model

All model files are saved to the shared Google Drive folder.

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/SmartDemand_Dataset/models/'
os.makedirs(MODEL_PATH, exist_ok=True)

# Save trained models
joblib.dump(rf_model,  MODEL_PATH + 'random_forest_model.joblib')
joblib.dump(lr_model,  MODEL_PATH + 'linear_regression_model.joblib')
joblib.dump(label_enc, MODEL_PATH + 'label_encoder.joblib')

# Save config (used by Streamlit app to reconstruct inputs correctly)
config = {
    'features'          : FEATURES,
    'target'            : TARGET,
    'categories'        : list(label_enc.classes_),
    'seasonality_index' : {int(k): float(v) for k, v in monthly_norm.items()},
    'model_metrics': {
        'random_forest': {
            'test_mae'  : round(float(mean_absolute_error(y_test, rf_pred)), 4),
            'test_rmse' : round(float(np.sqrt(mean_squared_error(y_test, rf_pred))), 4),
            'test_r2'   : round(float(r2_score(y_test, rf_pred)), 4)
        },
        'linear_regression': {
            'test_mae'  : round(float(mean_absolute_error(y_test, lr_pred)), 4),
            'test_rmse' : round(float(np.sqrt(mean_squared_error(y_test, lr_pred))), 4),
            'test_r2'   : round(float(r2_score(y_test, lr_pred)), 4)
        }
    }
}

with open(MODEL_PATH + 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved to:', MODEL_PATH)
print('  random_forest_model.joblib')
print('  linear_regression_model.joblib')
print('  label_encoder.joblib')
print('  config.json')
print()
print(f'Random Forest Test R2    : {config["model_metrics"]["random_forest"]["test_r2"]}')
print(f'Linear Regression Test R2: {config["model_metrics"]["linear_regression"]["test_r2"]}')

## Summary

Data integration, cleaning, feature engineering, EDA, model training, and evaluation are complete. Both Random Forest and Linear Regression models have been trained and saved to Google Drive.